In [1]:
!pip uninstall -y numpy
!pip install "numpy<2.0"

!pip install torch==2.2.2 torchvision==0.17.2 monai
!pip install scikit-image
!pip install wandb
!pip install import-ipynb
!pip install nibabel 


Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/lib/python3.10/shutil.py", line 816, in move
    os.rename(src, real_dst)
PermissionError: [Errno 13] Permission denied: '/usr/local/lib/python3.10/dist-packages/numpy-1.26.4.dist-info/' -> '/tmp/pip-uninstall-v1r7cg_b'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 107, in _run_wrapper
    status = _inner_run()
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 98, in _inner_run
    return self.run(options, args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/commands/uninstall.py", line 105, in run
    uninstall_pathset = req.uninstall(
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/req/req_install.py", line 675, in uninstall
    uninstalle

In [15]:
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
import monai
import torch
import re
import torchvision.transforms as transforms
from PIL import Image
from monai.transforms import LoadImage
import import_ipynb
from Functions import patients_dicts
from monai.data import MetaTensor

zip_file = "Resources.zip"
os.makedirs("train", exist_ok=True)
!unzip -o -q {zip_file} -d {"train"}
data_path = "train"

full_dict_list = patients_dicts(data_path)

In [16]:
print(len(full_dict_list))

100


In [17]:
import random

def split_dataset(dataset, per_train=16, per_val=4, seed=None):
    
    if seed is not None:
        random.seed(seed)
        
    catogories_numbers = {}
    catogories_ID = {}

    for patient in dataset:
        disease = patient["Disease"]
        if disease not in catogories_numbers:
            catogories_numbers[disease] = 0
            catogories_ID[disease] = []
        catogories_numbers[disease] += 1
        catogories_ID[disease].append(patient)
        
    split_train_dataset = []
    split_val_dataset = []
    
    for disease, patientIDS in catogories_ID.items():
        random.shuffle(patientIDS)

        train_ids = patientIDS[:per_train]
        val_ids = patientIDS[per_train:per_train+per_val]

        split_train_dataset.extend(train_ids)
        split_val_dataset.extend(val_ids)

    return split_train_dataset, split_val_dataset

In [18]:
from Functions import par_voxelsize, par_size
from monai.transforms import LoadImaged, Compose, EnsureChannelFirstd,ScaleIntensityd, Spacingd,LoadImaged, ResizeWithPadOrCropd

data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED","maskED","imgES","maskES"], channel_dim="no_channel")])

train_dataset_raw = monai.data.Dataset(full_dict_list, transform=data_transform)

voxel_size = []
mean_voxel, std_voxel, max_voxelsize = par_voxelsize(voxel_size,train_dataset_raw)

print("Median voxel size:", mean_voxel)
print("Std voxel size:", std_voxel)
print("Max voxel size:", max_voxelsize)

"""
sizes = []
max_size, listofsizes = par_size(sizes,train_dataset_raw)
print("Max image size:", max_size)
print(listofsizes)
"""

Median voxel size: [ 1.5625  1.5625 10.    ]
Std voxel size: [0.18463361 0.18463361 1.6644155 ]
Max voxel size: [ 1.91964  1.91964 10.     ]


'\nsizes = []\nmax_size, listofsizes = par_size(sizes,train_dataset_raw)\nprint("Max image size:", max_size)\nprint(listofsizes)\n'

In [19]:
from monai.transforms import LoadImaged, Compose, EnsureChannelFirstd,ScaleIntensityd, Spacingd,LoadImaged, ResizeWithPadOrCropd, NormalizeIntensityd, RandFlipd, RandRotated, RandGaussianSmoothd 

val_data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED","maskED","imgES","maskES"], channel_dim="no_channel"),
    Spacingd(keys=["imgED","maskED","imgES","maskES"], pixdim=(mean_voxel[0],mean_voxel[1],mean_voxel[2]), mode=("bilinear", "nearest","bilinear", "nearest"),
                ensure_same_shape=True,align_corners=False),
    ResizeWithPadOrCropd(keys=["imgED","maskED","imgES","maskES"],spatial_size=(256,256,16)), #added cause the images needed to be devided by 16 for the strides in the Unet image
    NormalizeIntensityd(keys=["imgED", "imgES"], nonzero=True, channel_wise=True)
])

train_data_transform = Compose([
    LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
    EnsureChannelFirstd(keys=["imgED","maskED","imgES","maskES"], channel_dim="no_channel"),
    Spacingd(keys=["imgED","maskED","imgES","maskES"], pixdim=(mean_voxel[0],mean_voxel[1],mean_voxel[2]), mode=("bilinear", "nearest","bilinear", "nearest"),
                ensure_same_shape=True,align_corners=False),
    ResizeWithPadOrCropd(keys=["imgED","maskED","imgES","maskES"],spatial_size=(256,256,16)),#added cause the images needed to be devided by 16 for the strides in the Unet image we can look into if we can solve this an other way
    NormalizeIntensityd(keys=["imgED", "imgES"], nonzero=True, channel_wise=True),
    RandFlipd(keys=["imgED", "maskED", "imgES", "maskES"], prob=0.33, spatial_axis=[0, 1, 2]),
    RandRotated(
        keys=["imgED", "maskED", "imgES", "maskES"],
        range_x=0.4, 
        prob=0.33,
        mode=["bilinear", "nearest", "bilinear", "nearest"]
    ),
    RandGaussianSmoothd(keys=["imgED", "imgES"], prob = 0.33)

])


train_dict_list, val_dict_list = split_dataset(full_dict_list)

val_dataset = monai.data.Dataset(val_dict_list, transform=val_data_transform)
train_dataset = monai.data.Dataset(train_dict_list, transform=train_data_transform)
#using the split function to split the data
train_dict_list, val_dict_list = split_dataset(full_dict_list, per_train=16, per_val=4, seed=None)
#
train_dataset = monai.data.Dataset(train_dict_list, transform=train_data_transform)
val_dataset   = monai.data.Dataset(val_dict_list, transform=val_data_transform)

train_loader = monai.data.DataLoader(train_dataset, batch_size=1, collate_fn=monai.data.pad_list_data_collate)
val_loader = monai.data.DataLoader(val_dataset, batch_size=1, collate_fn=monai.data.pad_list_data_collate)

print("Total dataset size", len(full_dict_list))

#for sample in train_dataset:
#    aff = sample["imgED"].meta["affine"]
#    spacing = np.sqrt((aff[:3, :3]**2).sum(0))
#    print( "spacing:", spacing, "shape:", tuple(sample["imgED"].shape))

Total dataset size 100


In [31]:
def visualize_heart_sample(sample, title=None):
    # Visualize the x-ray and overlay the mask, using the dictionary as input
    for i in range(2):
        if i == 0:
            image = np.squeeze(sample['imgED'])
            mask = np.squeeze(sample['maskED'])
        else:
            image = np.squeeze(sample['imgES'])
            mask = np.squeeze(sample['maskES'])

        plt.figure(figsize=[10,7])
        plt.imshow(image, 'gray')

        mask1 = np.ma.masked_where(mask != 1, mask)
        plt.imshow(mask1, 'Greens', alpha = 0.5, clim=[0,1], interpolation='nearest')

        mask2 = np.ma.masked_where(mask != 2, mask)
        plt.imshow(mask2, 'Reds', alpha = 0.5, clim=[0,1], interpolation='nearest')

        mask3 = np.ma.masked_where(mask != 3, mask)
        plt.imshow(mask3, 'Blues', alpha = 0.5, clim=[0,1], interpolation='nearest')
        if title is not None:
            plt.title(title)
        plt.show()

#shape = train_dataset[0]['imgED'].shape
#for i in range(shape[3]):
#    single_data = train_dataset[0]
#    sample_slice = {
#    'imgED': single_data['imgED'][0,:, :, i],
#    'maskED': single_data['maskED'][0,:, :, i],
#    'imgES': single_data['imgES'][0,:, :, i],
#    'maskES': single_data['maskES'][0,:, :, i]
#    }
#    visualize_heart_sample(sample_slice, title=f"patient:{single_data['ID']},disease:{single_data['Disease']},slice:{i}")

In [53]:
"""
#Validation Set
#look in to data_transform now themporary fix:
#data_transform = Compose([
from monai.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
#later on we look in to splitting dissease/non disease right for now random:
train_dataset,val_dataset = train_test_split(
    train_dict_list,
    test_size=0.2,   # 80% training 20% val
    random_state=42  # ensures reproducibility
)

train_dataset = Dataset(train_list, transform=train_transform)
val_dataset = Dataset(val_list, transform=val_transform)
#val_dataset = MedMNISTData(val_dict,transform=Load_my_image)
#train_dataset = MedMNISTData(train_dict, transform= Load_my_image)
#later on 

#validation_data = monai.data.CacheDataset(val_dict,transform=data_transform)
#train_data =monai.data.CacheDataset(train_dict,transform=data_transform) 
train_loader = monai.data.DataLoader(train_dataset,batch_size=1,collate_fn=monai.data.pad_list_data_collate)
validation_loader = monai.data.DataLoader(val_dataset,batch_size=1,collate_fn=monai.data.pad_list_data_collate) #adding patches basic monai function
"""



Train loader length: 3
Validation loader length: 1
Validation loader length: 1


In [49]:
#Defining the Unet model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'The used device is {device}')

model = monai.networks.nets.UNet(
    spatial_dims=3,
    in_channels=1, 
    out_channels=4, #since we want to segment 3 heart areas+ back
    channels=(8, 16, 32, 64, 128), #checking if the chanels are correct
    strides=(2, 2, 2, 2),
    #num_res_units=2,
).to(device)


The used device is cuda


In [50]:
#weight decays
epochs= 250 #looking if we want to change
batch = 1 
learning_rate = 1e-3
loss_function =  monai.losses.DiceLoss(softmax=True,to_onehot_y=True,batch=True) #Dice loss because segmentation
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) #look in to if we want to keep using Adam or if we want an other optimizer

In [58]:

import wandb
wandb.login()#make sure you're in the right prodject!! 

run = wandb.init(
    project='Deep_learning_project',
    name='All_patients_run_ED_ES_augmentatie',
    config={
        'loss function': str(loss_function), 
        'lr': optimizer.param_groups[0]["lr"],
        'transform': [str(t) for t in train_data_transform.transforms],
        'batch_size': train_loader.batch_size,
    }
)

def wandb_masks(mask_output, mask_gt):
    """Generate mask dictionary for W&B logging."""
    # Convert model output to class predictions
    mask_output = torch.softmax(mask_output, dim=1)  # shape: (B, C, H, W, D)
    mask_output = torch.argmax(mask_output, dim=1).cpu().numpy()

    # Convert ground truth to class indices if one-hot
    if mask_gt.shape[1] > 1:  # check if one-hot
        mask_gt = torch.argmax(mask_gt, dim=1)
    mask_gt = mask_gt.cpu().numpy()

    class_labels = {0: 'background', 1: 'right_ventricle', 2: 'left_ventricle', 3: 'myocardium'}
    masks = {
        'predictions': {'mask_data': mask_output, 'class_labels': class_labels},
        'ground_truth': {'mask_data': mask_gt, 'class_labels': class_labels}
    }
    return masks

    # Create list of images that have segmentation masks for model output and ground truth
def log_to_wandb(epoch, batch_data, outputs, frame='ED'):
    img = batch_data[f'img{frame}'][0].cpu().numpy()
    mask_gt = batch_data[f'mask{frame}'][0].float().to(device)
    mask_output = outputs
    wandb_img = wandb.Image(img, masks=wandb_masks(mask_output, mask_gt))
    wandb.log({f'results_{frame}': wandb_img, 'epoch': epoch})
    
    #log_to_wandb(epoch, loss.item(), None, batch, outputED, frame='ED')
    #log_to_wandb(epoch, loss.item(), None, batch, outputES, frame='ES')

# Store the network parameters        
torch.save(model.state_dict(), r'trainedUNet.pt')

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


In [63]:
import tqdm


def train_medmnistmodel(model, train_dataloader, val_dataloader, optimizer, epochs, val_freq=2,log_images=True):
    train_loss = []
    val_loss = []
    for epoch in tqdm.tqdm(range(epochs)):
        model.train()
        steps = 0
        epoch_loss = 0
        for batch in train_dataloader: #250 steps per epoch, figure out how many steps we need, going to the data once, early stopping if patient level =15
            optimizer.zero_grad() #backropagation
            imagED = batch['imgED'].float().to(device)
            labelsED = batch['maskED'].long().to(device)
            imagES = batch['imgES'].float().to(device)
            labelsES = batch['maskES'].long().to(device)
            outputED = model(imagED)
            outputES = model(imagES)
            loss = (loss_function(outputED, labelsED)+loss_function(outputES, labelsES))/2
            epoch_loss += loss.item()
            loss.backward()
            optimizer.step()
            steps += 1
            
        if log_images and steps % 50 == 0:
                log_to_wandb(epoch, batch, outputED, frame='ED')
                log_to_wandb(epoch, batch, outputES, frame='ES')
        avg_train_loss = epoch_loss / steps        
        train_loss.append(avg_train_loss)

        # validation loop trough 20 
        if epoch % val_freq == 0:
            val_steps = 0
            val_epoch_loss = 0
            model.eval()
            for batch in val_dataloader:
                imagED = batch['imgED'].float().to(device)
                labelED = batch['maskED'].float().to(device)
                imagES = batch['imgES'].float().to(device)
                labelES = batch['maskES'].float().to(device)
                outputED = model(imagED)
                outputES = model(imagES)
                loss = (loss_function(outputED, labelED)+loss_function(outputES,labelES))/2
                val_epoch_loss += loss.item()
                val_steps += 1
            avg_val_loss = val_epoch_loss / val_steps
            val_loss.append(avg_val_loss)
            wandb.log({'epoch': epoch,'train_loss': avg_train_loss,'val_loss': avg_val_loss})
            
    torch.save(model.state_dict(), 'trainedUNet.pt')
    wandb.finish()
    return train_loss, val_loss, model

In [64]:
#applying the training set
train_loss, val_loss, model = train_medmnistmodel(model, train_loader, val_loader, optimizer, epochs=epochs)

100%|██████████| 250/250 [1:09:01<00:00, 16.57s/it]


epoch,▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
train_loss,█▇▅▅▅▄▄▄▃▃▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,██▇▆▅▄▄▄▄▄▃▂▂▂▂▂▂▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,248
train_loss,0.23903
val_loss,0.27513
